# 🏋️ Sistem Klasifikasi Obesitas — Model + API

Notebook ini berisi pipeline lengkap:
1. **Data Preparation** — Feature Engineering + Encoding
2. **Scaler** — QuantileTransformer (`scaler_obesitas.pkl`)
3. **Multi-Model Training + Selection** — Model terbaik disimpan (`model_obesitas_best.pkl`)
4. **Flask API** — Endpoint `/predict` siap pakai

> ⚠️ Jalankan sel secara berurutan dari atas ke bawah.

---
## 📦 1. Install & Import Library

In [ ]:
# Install dependensi yang dibutuhkan untuk API
!pip install flask flask-ngrok pyngrok -q

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import QuantileTransformer, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import joblib

print('✅ Semua library berhasil di-import!')

---
## 📂 2. Load & Prepare Data

In [ ]:
# ─── GANTI PATH CSV SESUAI LOKASI FILE KAMU ───────────────────────
CSV_PATH = 'ObesityDataSet_raw_and_data_sinthetic.csv'
# ──────────────────────────────────────────────────────────────────

df = pd.read_csv(CSV_PATH)
print(f'Data loaded: {df.shape[0]} baris, {df.shape[1]} kolom')
df.head(3)

In [ ]:
# ── Cleaning dasar ─────────────────────────────────────────────────
df.drop_duplicates(inplace=True)

# Bulatkan float selain Height & Weight
for col in df.columns:
    if df[col].dtype == 'float64' and col not in ['Height', 'Weight']:
        df[col] = df[col].round().astype('int64')

# ── Feature Engineering ────────────────────────────────────────────
def feature_engineering(df):
    mapping_obesitas = {
        'Insufficient_Weight': 'Underweight',
        'Normal_Weight'      : 'Normal',
        'Overweight_Level_I' : 'Overweight',
        'Overweight_Level_II': 'Overweight',
        'Obesity_Type_I'     : 'Obesity',
        'Obesity_Type_II'    : 'Obesity',
        'Obesity_Type_III'   : 'Obesity',
    }
    df['Weight_Status']       = df['NObeyesdad'].replace(mapping_obesitas)
    df['BMI']                 = df['Weight'] / (df['Height'] ** 2)
    df['Water_Intake_Per_Kg'] = df['CH2O'] / df['Weight']
    df.drop(columns=['NObeyesdad'], inplace=True, errors='ignore')
    return df

df = feature_engineering(df)

# ── Encoding ───────────────────────────────────────────────────────
# Strip whitespace dulu
cat_cols = ['CALC','CAEC','Gender','FAVC','SCC','SMOKE',
            'family_history_with_overweight','MTRANS','Weight_Status']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].str.strip()

# Ordinal encoding
urutan_mtrans    = ['Bike','Walking','Public_Transportation','Motorbike','Automobile']
urutan_frekuensi = ['no','Sometimes','Frequently','Always']
urutan_target    = ['Underweight','Normal','Overweight','Obesity']

ord_enc = OrdinalEncoder(categories=[urutan_mtrans, urutan_frekuensi, urutan_frekuensi])
df[['MTRANS_Num','CAEC_Num','CALC_Num']] = ord_enc.fit_transform(df[['MTRANS','CAEC','CALC']])

ord_target = OrdinalEncoder(categories=[urutan_target])
df['Weight_Status_Num'] = ord_target.fit_transform(df[['Weight_Status']])

# Binary mapping
binary_map = {'no': 0, 'yes': 1}
for col in ['FAVC','SCC','SMOKE','family_history_with_overweight']:
    df[f'{col}_Num'] = df[col].map(binary_map)
df['Gender_Num'] = df['Gender'].map({'Male': 0, 'Female': 1})

print(f'✅ Feature engineering & encoding selesai — shape: {df.shape}')

---
## ✂️ 3. Feature Selection & Train/Test Split

In [ ]:
# Fitur yang digunakan (8 fitur terpilih)
FEATURES = [
    'Age', 'BMI', 'Water_Intake_Per_Kg', 'FAF',
    'family_history_with_overweight_Num',
    'CAEC_Num', 'FAVC_Num', 'SCC_Num'
]
TARGET = 'Weight_Status_Num'

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

# Split 80:20 dengan stratify
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train : {X_train.shape}  |  X_test : {X_test.shape}')
print(f'Distribusi target (train): {dict(y_train.value_counts().sort_index())}')

---
## ⚖️ 4. Scaler — QuantileTransformer

In [ ]:
# Kolom kontinu yang perlu di-scale
COLS_TO_SCALE = ['Age', 'BMI', 'Water_Intake_Per_Kg']

scaler = QuantileTransformer(output_distribution='normal', random_state=42)

X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()

X_train_scaled[COLS_TO_SCALE] = scaler.fit_transform(X_train[COLS_TO_SCALE])
X_test_scaled[COLS_TO_SCALE]  = scaler.transform(X_test[COLS_TO_SCALE])

# Simpan scaler
joblib.dump(scaler, 'scaler_obesitas.pkl')
print('✅ Scaler berhasil di-fit dan disimpan → scaler_obesitas.pkl')
print(f'   Fitur yang di-scale : {COLS_TO_SCALE}')

---
## 🤖 5. Multi-Model Training + Cross-Validation

In [ ]:
models = {
    'Logistic Regression' : LogisticRegression(max_iter=1000, random_state=42),
    'K-Nearest Neighbors' : KNeighborsClassifier(n_neighbors=7),
    'Decision Tree'       : DecisionTreeClassifier(random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=200, random_state=42),
    'Extra Trees'         : ExtraTreesClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=200, random_state=42),
    'SVM (RBF)'           : SVC(kernel='rbf', C=1, probability=True, random_state=42),
    'Naive Bayes'         : GaussianNB(),
}

cv      = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1_macro', 'f1_weighted']

results = {}
print(f"{'Model':<25} {'Accuracy':>10} {'F1-Macro':>10} {'F1-Weighted':>12} {'Std Acc':>9}")
print('-' * 70)

for name, model in models.items():
    cv_res = cross_validate(model, X_train_scaled, y_train, cv=cv, scoring=scoring)
    acc  = cv_res['test_accuracy'].mean()
    std  = cv_res['test_accuracy'].std()
    f1m  = cv_res['test_f1_macro'].mean()
    f1w  = cv_res['test_f1_weighted'].mean()
    results[name] = {'accuracy': acc, 'std': std, 'f1_macro': f1m, 'f1_weighted': f1w}
    print(f'{name:<25} {acc:>10.4f} {f1m:>10.4f} {f1w:>12.4f} {std:>9.4f}')

df_results = pd.DataFrame(results).T.sort_values('f1_weighted', ascending=False)

best_model_name = df_results['f1_weighted'].idxmax()
print(f'\n🏆 Model Terbaik: {best_model_name} | F1-Weighted: {df_results.loc[best_model_name, "f1_weighted"]:.4f}')

---
## 🏆 6. Final Training & Simpan Model Terbaik

In [ ]:
# Training final model terbaik menggunakan SELURUH data train
best_model = models[best_model_name]
best_model.fit(X_train_scaled, y_train)

# Evaluasi di test set
y_pred = best_model.predict(X_test_scaled)

label_names = ['Underweight', 'Normal', 'Overweight', 'Obesity']
print(f'Model   : {best_model_name}')
print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print(f'F1-Macro: {f1_score(y_test, y_pred, average="macro"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=label_names))

# Simpan model
joblib.dump(best_model, 'model_obesitas_best.pkl')
print(f'✅ Model terbaik disimpan → model_obesitas_best.pkl')

---
## 🔍 7. Verifikasi File Tersimpan

In [ ]:
import os

files_to_check = ['scaler_obesitas.pkl', 'model_obesitas_best.pkl']
print('File yang tersimpan:')
for f in files_to_check:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'  ✅ {f:<35} ({size:.1f} KB)')
    else:
        print(f'  ❌ {f} TIDAK DITEMUKAN!')

# Verifikasi load ulang
test_scaler = joblib.load('scaler_obesitas.pkl')
test_model  = joblib.load('model_obesitas_best.pkl')
print(f'\nVerifikasi load:')
print(f'  Scaler type : {type(test_scaler).__name__}')
print(f'  Model type  : {type(test_model).__name__}')

---
## 🌐 8. Flask API

### Input yang dibutuhkan (JSON):
```json
{
  "Age": 30,
  "Height": 1.72,
  "Weight": 85,
  "CH2O": 2.5,
  "FAF": 1,
  "family_history_with_overweight": "yes",
  "CAEC": "Sometimes",
  "FAVC": "yes",
  "SCC": "no"
}
```
### Endpoint:
- `GET  /health` → cek status API
- `POST /predict` → prediksi Weight Status

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# FLASK API — Tulis ke file app.py, lalu jalankan di thread terpisah
# ─────────────────────────────────────────────────────────────────────

api_code = '''
import numpy as np
import pandas as pd
import joblib
from flask import Flask, request, jsonify

# ── Konstanta ─────────────────────────────────────────────────────────
COLS_TO_SCALE   = ["Age", "BMI", "Water_Intake_Per_Kg"]
FEATURE_ORDER   = ["Age", "BMI", "Water_Intake_Per_Kg", "FAF",
                   "family_history_with_overweight_Num",
                   "CAEC_Num", "FAVC_Num", "SCC_Num"]
CLASS_NAMES     = ["Underweight", "Normal", "Overweight", "Obesity"]

# Mapping untuk input mentah dari user
CAEC_MAP  = {"no": 0, "Sometimes": 1, "Frequently": 2, "Always": 3}
BINARY_MAP = {"no": 0, "yes": 1}

# ── Load scaler & model ───────────────────────────────────────────────
scaler = joblib.load("scaler_obesitas.pkl")
model  = joblib.load("model_obesitas_best.pkl")

app = Flask(__name__)

# ── Helper: preprocess raw input ──────────────────────────────────────
def preprocess(data: dict) -> pd.DataFrame:
    """
    Menerima input mentah dari user, menghitung fitur turunan,
    melakukan encoding, dan mengembalikan DataFrame siap prediksi.
    """
    # Hitung BMI dan Water Intake Per Kg dari Height/Weight/CH2O
    height = float(data["Height"])
    weight = float(data["Weight"])
    ch2o   = float(data["CH2O"])

    bmi                 = weight / (height ** 2)
    water_intake_per_kg = ch2o / weight

    row = {
        "Age"                                : float(data["Age"]),
        "BMI"                                : bmi,
        "Water_Intake_Per_Kg"                : water_intake_per_kg,
        "FAF"                                : float(data["FAF"]),
        "family_history_with_overweight_Num" : BINARY_MAP.get(str(data["family_history_with_overweight"]).strip().lower(), 0),
        "CAEC_Num"                           : CAEC_MAP.get(str(data["CAEC"]).strip(), 1),
        "FAVC_Num"                           : BINARY_MAP.get(str(data["FAVC"]).strip().lower(), 0),
        "SCC_Num"                            : BINARY_MAP.get(str(data["SCC"]).strip().lower(), 0),
    }

    df_input = pd.DataFrame([row])[FEATURE_ORDER]

    # Scale kolom kontinu
    df_input[COLS_TO_SCALE] = scaler.transform(df_input[COLS_TO_SCALE])

    return df_input


# ── Endpoint: Health Check ────────────────────────────────────────────
@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status" : "ok",
        "model"  : type(model).__name__,
        "scaler" : type(scaler).__name__,
        "classes": CLASS_NAMES
    })


# ── Endpoint: Predict ─────────────────────────────────────────────────
@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json(force=True)
        if data is None:
            return jsonify({"error": "Request body harus berupa JSON"}), 400

        # Validasi field wajib
        required = ["Age", "Height", "Weight", "CH2O", "FAF",
                    "family_history_with_overweight", "CAEC", "FAVC", "SCC"]
        missing = [f for f in required if f not in data]
        if missing:
            return jsonify({"error": f"Field berikut wajib diisi: {missing}"}), 400

        # Preprocess
        df_input = preprocess(data)

        # Prediksi
        pred_idx   = int(model.predict(df_input)[0])
        pred_label = CLASS_NAMES[pred_idx]

        # Probabilitas (jika model support)
        if hasattr(model, "predict_proba"):
            proba = model.predict_proba(df_input)[0]
            probabilities = {cls: round(float(p), 4) for cls, p in zip(CLASS_NAMES, proba)}
        else:
            probabilities = None

        response = {
            "predicted_class" : pred_label,
            "class_index"     : pred_idx,
            "probabilities"   : probabilities,
            "input_processed" : {
                "BMI"                : round(float(data["Weight"]) / (float(data["Height"]) ** 2), 2),
                "Water_Intake_Per_Kg": round(float(data["CH2O"]) / float(data["Weight"]), 4),
            }
        }

        return jsonify(response)

    except KeyError as e:
        return jsonify({"error": f"Field tidak valid: {str(e)}"}), 400
    except Exception as e:
        return jsonify({"error": str(e)}), 500


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, debug=False)
'''

with open('app.py', 'w') as f:
    f.write(api_code)

print('✅ File app.py berhasil ditulis!')
print('   Endpoint:')
print('   • GET  /health  → cek status')
print('   • POST /predict → prediksi Weight Status')

---
## 🚀 9. Jalankan API

Pilih salah satu cara di bawah ini:

In [ ]:
# ── CARA 1: Jalankan Flask di thread (untuk Google Colab / Jupyter) ──
import threading
import subprocess

def run_flask():
    subprocess.run(['python', 'app.py'])

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()

import time
time.sleep(2)  # Tunggu Flask ready
print('✅ Flask API berjalan di http://localhost:5000')

In [ ]:
# ── CARA 2 (Opsional): Expose via ngrok untuk public URL ─────────────
# Uncomment dan isi token ngrok kamu dari https://dashboard.ngrok.com

# from pyngrok import ngrok
# ngrok.set_auth_token("ISI_TOKEN_NGROK_KAMU_DISINI")
# public_url = ngrok.connect(5000)
# print(f'🌐 Public URL: {public_url}')
# print(f'   Predict endpoint: {public_url}/predict')

print('💡 Uncomment sel ini untuk expose ke internet via ngrok')

---
## 🧪 10. Test API

In [ ]:
import requests
import json

BASE_URL = 'http://localhost:5000'

# ── Test Health Check ─────────────────────────────────────────────────
resp = requests.get(f'{BASE_URL}/health')
print('── Health Check ──')
print(json.dumps(resp.json(), indent=2))
print()

In [ ]:
# ── Test Predict: Sampel 1 — Risiko Obesity ───────────────────────────
sample_obesity = {
    "Age"                           : 35,
    "Height"                        : 1.68,
    "Weight"                        : 92,
    "CH2O"                          : 2.0,
    "FAF"                           : 0,
    "family_history_with_overweight": "yes",
    "CAEC"                          : "Frequently",
    "FAVC"                          : "yes",
    "SCC"                           : "no"
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_obesity)
result = resp.json()

print('── Prediksi Sampel 1 (Risiko Obesity) ──')
print(f'  Prediksi  : {result["predicted_class"]}')
print(f'  BMI       : {result["input_processed"]["BMI"]}')
if result['probabilities']:
    print('  Probabilitas:')
    for cls, prob in result['probabilities'].items():
        bar = '█' * int(prob * 30)
        print(f'    {cls:<15}: {prob:.4f}  {bar}')

In [ ]:
# ── Test Predict: Sampel 2 — Profil Normal ────────────────────────────
sample_normal = {
    "Age"                           : 24,
    "Height"                        : 1.75,
    "Weight"                        : 68,
    "CH2O"                          : 2.8,
    "FAF"                           : 3,
    "family_history_with_overweight": "no",
    "CAEC"                          : "Sometimes",
    "FAVC"                          : "no",
    "SCC"                           : "yes"
}

resp = requests.post(f'{BASE_URL}/predict', json=sample_normal)
result = resp.json()

print('── Prediksi Sampel 2 (Profil Normal) ──')
print(f'  Prediksi  : {result["predicted_class"]}')
print(f'  BMI       : {result["input_processed"]["BMI"]}')
if result['probabilities']:
    print('  Probabilitas:')
    for cls, prob in result['probabilities'].items():
        bar = '█' * int(prob * 30)
        print(f'    {cls:<15}: {prob:.4f}  {bar}')

---
## 📋 Ringkasan File Output

| File | Keterangan |
|------|------------|
| `scaler_obesitas.pkl` | QuantileTransformer — fit dari data training |
| `model_obesitas_best.pkl` | Model terbaik hasil CV |
| `app.py` | Flask API (`/health` & `/predict`) |

### Cara Kirim Request Manual (cURL):
```bash
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "Age": 30, "Height": 1.70, "Weight": 80,
    "CH2O": 2.0, "FAF": 1,
    "family_history_with_overweight": "yes",
    "CAEC": "Sometimes", "FAVC": "yes", "SCC": "no"
  }'
```